# Notebook 2 — Preprocessing

This notebook covers label encoding, feature selection based on correlation analysis, and Min-Max scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

%matplotlib inline

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/ipl_dataset.csv')
data = df.copy()
print(data.shape)

## 2. Label Encoding

Machine learning models require numerical input. We encode all categorical columns using `LabelEncoder` and save each encoder so we can reuse it during inference.

In [ ]:
cat_cols = ['bat_team', 'bowl_team', 'venue', 'batsman', 'bowler']

data_encoded = data.copy()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    data_encoded[col] = le.fit_transform(data_encoded[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded '{col}': {len(le.classes_)} unique values")

data_encoded.head()

## 3. Correlation Heatmap

We compute pairwise Pearson correlations to identify redundant features.

In [ ]:
data_corr = data_encoded.drop(columns=['date', 'mid'])

plt.figure(figsize=(14, 10))
sns.heatmap(
    data_corr.corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150)
plt.show()

**Observation:** `runs_last_5`, `wickets_last_5`, and `non_striker` show high correlations with other features. These are dropped to reduce redundancy.

## 4. Feature Selection

In [ ]:
feature_cols = [
    'bat_team', 'bowl_team', 'venue',
    'runs', 'wickets', 'overs',
    'striker', 'batsman', 'bowler'
]

X = data_encoded[feature_cols]
y = data_encoded['total']

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape        : {y.shape}')

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')

## 6. Feature Scaling

Neural networks train better when all features are on the same scale. We use Min-Max scaling [0, 1]. The scaler is fit only on training data.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaled X_train range:', X_train_scaled.min().round(3), 'to', X_train_scaled.max().round(3))
print('Scaled X_test  range:', X_test_scaled.min().round(3),  'to', X_test_scaled.max().round(3))

In [ ]:
# Save for use in next notebook
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(label_encoders, '../models/label_encoders.pkl')
joblib.dump(scaler,         '../models/scaler.pkl')
np.save('../models/X_train_scaled.npy', X_train_scaled)
np.save('../models/X_test_scaled.npy',  X_test_scaled)
np.save('../models/y_train.npy',        y_train.values)
np.save('../models/y_test.npy',         y_test.values)
print('Preprocessing artifacts saved to models/')